# 02 - Phase 2 : Baselines (Benign / DDoS)

## Question
Avant de comparer quoi que ce soit de complexe : quelle est la performance d’un dummy, d’un modèle linéaire et d’un ensemble d’arbres sur la tâche binaire, avec la séparation sans fuite ?

## Protocole
- CIC-DDoS2019 après déduplication : 77 features, répartition en 80/20
- `DummyClassifier(strategy="stratified")` est le palier minimal.
- `LogisticRegression(max_iter=2000, class_weight="balanced")` est la référence linéaire.
- `RandomForestClassifier(n_estimators=300, class_weight="balanced")` est la référence non linéaire.
- La mise à l'échelle et la suppression des caractéristiques constantes sont intégrées au pipeline et ne s'appliquent qu'au modèle d'entraînement.
- Metrics: Macro-F1, Weighted-F1, per-class Precision/Recall/F1, PR-AUC, ROC-AUC, FPR at 95 % recall. L'accuracy est affichée mais n’est pas utilisée pour conclure.

Les nombres ci-dessous sont produits par `uv run daf-projet baseline` ; ce notebook charge `results/baseline_binary.csv` afin de ne pas réentraîner silencieusement sur un échantillon différent.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from daf_projet.data import group_attack_families, split
from daf_projet.experiments.common import load_deduped

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
RES = ROOT / "results"
FIG = ROOT / "figures"

results = pd.read_csv(RES / "baseline_binary.csv")
per_class = pd.read_csv(RES / "baseline_binary_per_class.csv")
display(results[["model", "macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95", "fit_seconds", "n_features_effective"]].round(4))
print("sklearn:", results["sklearn_version"].unique().tolist())

In [ ]:
metrics_long = results.melt(
    id_vars=["model"],
    value_vars=["macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr"],
    var_name="metric",
    value_name="value",
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=metrics_long, x="metric", y="value", hue="model", ax=ax)
ax.set_ylim(0, 1.02)
ax.set_title("Baseline binaire : le palier se sépare immédiatement des modèles")
ax.set_xlabel("")
ax.legend(title="", ncol=3, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "02_baseline_metrics.png", bbox_inches="tight")
plt.show()

display(
    results[["model", "macro_f1", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95"]]
    .set_index("model")
    .round(4)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, model in zip(axes, ["dummy", "logreg", "random_forest"]):
    matrix = pd.read_csv(RES / f"baseline_confusion_{model}.csv", index_col=0)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(model)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
fig.suptitle("Matrice de confusion - binaire Benign / DDoS", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "02_baseline_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# L'alignement des labels : sur l’ensemble des données, le dummy stratifié atteint
# Maccro-F1 ~ 1/13, la chance pour 13 classes.  La précision suit sum(prior^2), pas 1/13, car les classes sont déséquilibrées.  Une exécution limitée peut donner un Macro-F1 différent du dummy car la limite change les priorités de classe.
X, y_family, _, _ = load_deduped()
X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=42)
y_train = group_attack_families(y_family_train)
y_test = group_attack_families(y_family_test)

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)
pred = dummy.predict(X_test)
prior = y_train.value_counts(normalize=True)
print(f"classes : {y_train.nunique()}")
print(f"1 / n_classes : {1 / y_train.nunique():.4f}")
print(f"sum(prior^2) (accuracy): {(prior**2).sum():.4f}")
print(f"Dummy accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"Dummy Macro-F1 : {f1_score(y_test, pred, average='macro', zero_division=0):.4f}")
print("Les labels sont alignés : la précision du dummy suit sum(prior^2), pas un label regroupé.")

## Interprétation des résultats

**Vérification.** Sur l’ensemble des données, la variable fictive à 13 classes atteint un Macro-F1 ≈ 0,076 ≈
1/13, et sa précision suit sum(prior^2) ≈ 0,208.  Les étiquettes sont alignées. Le
seuil minimal binaire est ≈ 0,50, comme prévu pour un test à deux classes déséquilibré.

**Observation.** La régression logistique et la forêt aléatoire dépassent toutes deux 0,99
de Macro-F1 sur la tâche binaire ; le seuil minimal est d’environ 0,5 car l’ensemble de test
est déséquilibré en faveur des attaques.  L’écart entre les modèles est minime par rapport à l’
écart par rapport au seuil.

**Interprétation ML.** Le problème « Benign/DDoS » est facile à résoudre sur cet ensemble de données : une
frontière linéaire suffit déjà à le capturer.  Un score binaire quasi parfait n’est donc pas
la preuve que le modèle comprend les *types* d’attaques.  L’expérience 1 ne modifie que
la cible et conserve les mêmes lignes, les mêmes modèles et la même graine.

**Interprétation cyber.** Un détecteur binaire avec un taux de faux positifs (FPR) inférieur à 1% est utile sur le plan opérationnel
en tant qu’alerte de premier niveau, mais il ne peut pas indiquer à un analyste quelle mesure d’atténuation
appliquer.  La question suivante est de savoir si cette même représentation des flux reste valable lorsque
l’ensemble des étiquettes passe à 13 classes.

**Limites.** L’ensemble de test est un échantillon aléatoire issu d’une capture en laboratoire réalisée en 2019.  Il ne
mesure ni la dérive, ni l’évasion adversaire, ni le trafic chiffré.  La précision est indiquée,
mais les conclusions reposent sur le Macro-F1, le rappel par classe et le FPR.